<a href="https://colab.research.google.com/github/lmassaron/fine-tuning-workshop/blob/main/06_generate_sentiment_explanations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generating Financial Sentiment Explanations

In this notebook, we move beyond simple sentiment labels to create a **reasoned dataset**. We take our augmented dataset (original and synthetic headlines) and generate high-quality financial explanations for each.

### Why generate explanations?
- **Chain-of-Thought (CoT) Fine-tuning**: Training a model to reason before providing a label significantly improves its accuracy and robustness.
- **Transparency**: In financial applications, an "explainable" AI is more trustworthy for human analysts.
- **Data Augmentation**: We use a powerful teacher model (**Qwen 2.5 7B**) to "teach" a smaller student model how to think like a senior financial analyst.

### Technical Approach
We utilize **4-bit NormalFloat (NF4) quantization** to run a 7-billion parameter model on a single consumer GPU, ensuring high performance with manageable hardware requirements.

In [1]:
%%capture
import os

if "COLAB_" in "".join(os.environ.keys()):
    !pip install -U transformers trl accelerate bitsandbytes

In [2]:
import torch
from datasets import load_from_disk, Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings("ignore")

## 1. Configuration

We set up our model ID and quantization parameters. We use a larger batch size here as generating a single explanation is less memory-intensive than generating multiple variations.

In [3]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
DATASET_ID = "data/FinancialPhraseBank_augmented_judged"
OUTPUT_PATH = "data/FinancialPhraseBank_explained"
HF_OUTPUT_REPO = "lmassaron/FinancialPhraseBank_explained"

BATCH_SIZE = 16
MAX_NEW_TOKENS = 512
DEMO = False

QUANTIZATION_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

LABEL_MAP = {0: "negative", 1: "neutral", 2: "positive"}

## 2. Core Utilities and Prompt Engineering

We reuse the same core utilities from the previous step to maintain consistency and reduce code duplication.

In [4]:
def load_model(model_id: str):
    print(f"Loading {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=torch.float16,
        attn_implementation="sdpa",
        quantization_config=QUANTIZATION_CONFIG,
        device_map="auto",
    )
    model.eval()
    return tokenizer, model


def generate_batch(tokenizer, model, chats):
    inputs = tokenizer(
        chats, return_tensors="pt", padding=True, truncation=True, max_length=1024
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)


def generate_responses(tokenizer, model, system_prompt, user_prompts):
    chats = [
        tokenizer.apply_chat_template(
            [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": up},
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        for up in user_prompts
    ]
    return generate_batch(tokenizer, model, chats)


EXPLAINER_SYSTEM_PROMPT = (
    "You are a senior financial analyst with deep expertise in equity markets, "
    "corporate finance, and macroeconomics. "
    "Your task is to explain, in 2-4 sentences, why a financial news headline "
    "carries a specific market sentiment. "
    "Focus strictly on the financial implications: how the news affects revenue, "
    "profitability, cash flow, investor confidence, or market positioning. "
    "Be concise and precise. Do not repeat the sentence verbatim."
)


def build_explain_prompt(sentence: str, sentiment: str) -> str:
    return (
        f'Financial news headline:\n"{sentence}"\n\n'
        f"This headline has been classified as **{sentiment}** sentiment "
        f"from a financial markets perspective.\n\n"
        f"Explain why, focusing on the financial implications for investors, "
        f"the company, or the broader market."
    )

## 3. Loading the Teacher Model

In [5]:
tokenizer, model = load_model(MODEL_ID)

Loading Qwen/Qwen2.5-7B-Instruct...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

## 4. Inference Pipeline

We use the shared `generate_responses` utility to process our headlines in batches.

In [ ]:
def process_dataset(tokenizer, model, dataset, limit=None):
    results = []
    sentences = dataset["sentence"]
    labels = dataset["label"]
    is_augmented = dataset["is_augmented"]
    judge_score = dataset["judge_score"]

    if limit:
        sentences = sentences[:limit]
        labels = labels[:limit]

    for start in tqdm(
        range(0, len(sentences), BATCH_SIZE), desc="Generating explanations"
    ):
        batch_sentences = sentences[start : start + BATCH_SIZE]
        batch_labels = labels[start : start + BATCH_SIZE]

        explain_prompts = [
            build_explain_prompt(sent, LABEL_MAP[lbl])
            for sent, lbl in zip(batch_sentences, batch_labels)
        ]

        explanations = generate_responses(
            tokenizer, model, EXPLAINER_SYSTEM_PROMPT, explain_prompts
        )

        for i, (sentence, label, explanation) in enumerate(
            zip(batch_sentences, batch_labels, explanations)
        ):
            idx = start + i
            results.append(
                {
                    "sentence": sentence,
                    "label": label,
                    "sentiment": LABEL_MAP[label],
                    "explanation": explanation.strip(),
                    "is_augmented": is_augmented[idx],
                    "judge_score": judge_score[idx],
                }
            )
    return results

## 5. Execution

We load the augmented dataset from the previous notebook and generate explanations for all examples (original and synthetic).

In [7]:
print(f"Loading augmented dataset from: {DATASET_ID}")
raw = load_from_disk(DATASET_ID)

output_splits = {}
for split_name, data in raw.items():
    print(f"\n── Processing split: {split_name} ──")
    results = process_dataset(tokenizer, model, data, limit=5 if DEMO else None)
    output_splits[split_name] = Dataset.from_list(results)

output_ds = DatasetDict(output_splits)
output_ds.save_to_disk(OUTPUT_PATH)
print(f"\nDataset saved to: {OUTPUT_PATH}")

Loading augmented dataset from: data/FinancialPhraseBank_augmented_judged

── Processing split: train ──


Generating explanations:   0%|          | 0/494 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



── Processing split: validation ──


Generating explanations:   0%|          | 0/62 [00:00<?, ?it/s]

...

Generating explanations:   0%|          | 0/62 [00:00<?, ?it/s]

Saving the dataset (0/1 shards):   0%|          | 0/7902 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/988 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/985 [00:00<?, ? examples/s]


Dataset saved to: data/FinancialPhraseBank_explained


## 6. Verification & HF Export

In [8]:
ex = output_ds["train"][0]
print(f"Sentence  : {ex['sentence']}")
print(f"Sentiment : {ex['sentiment']}")
print(f"Augmented : {ex['is_augmented']}")
print(f"Explanation: {ex['explanation']}")

if not DEMO:
    print(f"Pushing to Hugging Face Hub: {HF_OUTPUT_REPO}")
    output_ds.push_to_hub(HF_OUTPUT_REPO, private=False)

Sentence  : The shares carry a right to dividend and other shareholder rights as from their registration with the Finnish Trade Register .
Sentiment : neutral
Augmented : False
Explanation: The inclusion of a right to dividend and other shareholder rights indicates that the shares have reached a status where they are eligible for dividends and certain voting rights. This typically does not directly impact revenue or profitability but can enhance investor confidence by providing clear entitlements and potentially increasing the perceived value of the shares, thus maintaining a neutral sentiment in terms of market reaction.
Pushing to Hugging Face Hub: lmassaron/FinancialPhraseBank_explained


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

Setting num_proc from 1 back to 1 for the test split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            